# 03. Lab Raw (검사 결과 원본 추출)

## 목적
코호트 환자의 검사 결과를 **1시간 단위**로 추출 (슬라이딩 윈도우 적용 전)

## 입력
- `cohort_base.csv`: 기본 코호트 (환자 목록)

## 출력
- `lab_raw.csv`: 시간별 검사 결과 (1 row = 1 patient × 1 hour)

## 전처리 범위
- [x] 데이터 추출 (labevents)
- [x] 집계 규칙 적용 (MIN/MAX/MEDIAN)
- [x] 이상치 클리핑
- [ ] 결측치 처리 → preprocessing 단계에서

## Item IDs 및 집계 규칙
| 분류 | 항목 | Item ID | 집계 | 임상적 사유 |
|------|------|---------|------|-------------|
| 산소/산도 | SaO2 | 50817 | MIN | 낮을수록 위험 |
| 산소/산도 | pH | 50820 | MIN | 낮을수록 산증 |
| 대사/염증 | Lactate | 50813 | MAX | 높을수록 위험 |
| 대사/염증 | Creatinine | 50912 | MAX | 높을수록 신부전 |
| 대사/염증 | Bilirubin | 50885 | MAX | 높을수록 간부전 |
| 혈액 | WBC | 51301 | MEDIAN | 안정적 대표값 |
| 혈액 | Platelets | 51265 | MEDIAN | 안정적 대표값 |
| 전해질 | Potassium | 50971 | MEDIAN | 안정적 대표값 |
| 전해질 | Sodium | 50983 | MEDIAN | 안정적 대표값 |

In [ ]:
import duckdb
import pandas as pd
import os

# 설정
DB_PATH = '../data/duckdb/mimic_total.duckdb'
INPUT_DIR = '../data/processed'
OUTPUT_DIR = '../data/processed'

con = duckdb.connect(DB_PATH)
print("=== 03. Lab Raw 추출 시작 ===")

## Step 1: 코호트 로드

In [ ]:
print("Step 1: 코호트 로드")

cohort_path = os.path.join(INPUT_DIR, 'cohort_base.csv')
df_cohort = pd.read_csv(cohort_path, parse_dates=['intime', 'outtime'])

print(f"✓ 코호트 로드 완료: {len(df_cohort):,}명")

con.register('cohort', df_cohort)

## Step 2: Lab 데이터 추출

In [ ]:
print("\nStep 2: Lab 데이터 추출 (1시간 단위)")

lab_query = """
WITH base_labs AS (
    SELECT
        c.stay_id,
        date_trunc('hour', CAST(l.charttime AS TIMESTAMP)) as charttime_h,
        l.itemid,
        TRY_CAST(l.valuenum AS DOUBLE) as val_num
    FROM cohort c
    INNER JOIN labevents l ON c.subject_id = l.subject_id
    WHERE l.itemid IN (
        '50817',  -- SaO2
        '50820',  -- pH
        '50813',  -- Lactate
        '50912',  -- Creatinine
        '50885',  -- Bilirubin
        '51301',  -- WBC
        '51265',  -- Platelets
        '50971',  -- Potassium
        '50983'   -- Sodium
    )
    AND TRY_CAST(l.valuenum AS DOUBLE) IS NOT NULL
    -- ICU 체류 기간 내 데이터만
    AND CAST(l.charttime AS TIMESTAMP) >= c.intime
    AND CAST(l.charttime AS TIMESTAMP) <= c.outtime
),
aggregated AS (
    SELECT
        stay_id,
        charttime_h,
        
        -- [산소/산도] 최저값 (MIN)
        MIN(CASE WHEN itemid = '50817' THEN val_num END) as sao2,
        MIN(CASE WHEN itemid = '50820' THEN val_num END) as ph,
        
        -- [대사/염증] 최고값 (MAX)
        MAX(CASE WHEN itemid = '50813' THEN val_num END) as lactate,
        MAX(CASE WHEN itemid = '50912' THEN val_num END) as creatinine,
        MAX(CASE WHEN itemid = '50885' THEN val_num END) as bilirubin,
        
        -- [혈액/전해질] 중간값 (MEDIAN)
        MEDIAN(CASE WHEN itemid = '51301' THEN val_num END) as wbc,
        MEDIAN(CASE WHEN itemid = '51265' THEN val_num END) as platelets,
        MEDIAN(CASE WHEN itemid = '50971' THEN val_num END) as potassium,
        MEDIAN(CASE WHEN itemid = '50983' THEN val_num END) as sodium
        
    FROM base_labs
    GROUP BY stay_id, charttime_h
)
SELECT
    stay_id,
    charttime_h,
    
    -- 이상치 클리핑
    CASE WHEN sao2 BETWEEN 50 AND 100 THEN sao2 ELSE NULL END as sao2,
    CASE WHEN ph BETWEEN 6.8 AND 7.8 THEN ph ELSE NULL END as ph,
    CASE WHEN lactate BETWEEN 0.1 AND 30 THEN lactate ELSE NULL END as lactate,
    CASE WHEN creatinine BETWEEN 0.1 AND 30 THEN creatinine ELSE NULL END as creatinine,
    CASE WHEN bilirubin BETWEEN 0.1 AND 50 THEN bilirubin ELSE NULL END as bilirubin,
    CASE WHEN wbc BETWEEN 0.1 AND 100 THEN wbc ELSE NULL END as wbc,
    CASE WHEN platelets BETWEEN 5 AND 1000 THEN platelets ELSE NULL END as platelets,
    CASE WHEN potassium BETWEEN 1.5 AND 10 THEN potassium ELSE NULL END as potassium,
    CASE WHEN sodium BETWEEN 110 AND 170 THEN sodium ELSE NULL END as sodium
    
FROM aggregated
ORDER BY stay_id, charttime_h
"""

print("  쿼리 실행 중...")
df_lab = con.execute(lab_query).df()

print(f"✓ Lab 추출 완료")
print(f"  - 총 행 수: {len(df_lab):,}개")
print(f"  - 고유 환자: {df_lab['stay_id'].nunique():,}명")

## Step 3: 데이터 품질 확인

In [ ]:
print("\nStep 3: 데이터 품질 확인")

lab_cols = ['sao2', 'ph', 'lactate', 'creatinine', 'bilirubin', 'wbc', 'platelets', 'potassium', 'sodium']

print("\n=== 결측치 비율 ===")
for col in lab_cols:
    missing_rate = df_lab[col].isna().mean() * 100
    print(f"  {col}: {missing_rate:.1f}% 결측")

print("\n=== 기술 통계 ===")
df_lab[lab_cols].describe().round(2)

## Step 4: 저장

In [ ]:
print("\n" + "="*60)
print("CSV 저장")
print("="*60)

output_path = os.path.join(OUTPUT_DIR, 'lab_raw.csv')
df_lab.to_csv(output_path, index=False)

file_size = os.path.getsize(output_path) / (1024 * 1024)

print(f"\n✓ 저장 완료: lab_raw.csv")
print(f"  - 파일 크기: {file_size:.2f} MB")
print(f"  - 행 수: {len(df_lab):,}개")
print(f"  - 경로: {output_path}")

In [ ]:
print("\n=== 샘플 데이터 ===")
df_lab.head(10)

In [ ]:
con.close()
print("\n=== 03. Lab Raw 추출 완료 ===")